In [8]:
%pip install pymongo
from pymongo import MongoClient

Note: you may need to restart the kernel to use updated packages.


In [9]:
client = MongoClient("mongodb://localhost:27017/")
client.admin.command('ping')

{'ok': 1.0}

In [12]:
db = client['schoolDB']
students = db['students']

In [15]:
# 2. Insert Document
student = db.students.insert_one({
    "name": "John Doe",
    "age": 20,
    "dept":"CSE", 
    "marks":98,
    "skills":["Python","Java"],
    "city":"hyderabad",
    "fees": {
        "total": 50000,
        "paid": 45000
    }
    })

In [16]:
# 3. Read with Filter
results = db.students.find({"marks" : {"$gte": 90}},{"name":1,"_id":0,"marks":1})
for result in results:
    print(result)

{'name': 'Aarav', 'marks': 140}
{'name': 'Kabir', 'marks': 146}
{'name': 'Rohan', 'marks': 133}
{'name': 'Vikram', 'marks': 110}
{'name': 'Mourya', 'marks': 149}
{'name': 'John Doe', 'marks': 98}


In [18]:
# 4. Projection + Sorting
results = db.students.find({},{"name":1 ,"dept":1,"_id":0, "marks":1}).sort("marks", -1).limit(3)
for result in results:
    print(result)

{'name': 'Mourya', 'marks': 149}
{'name': 'Kabir', 'marks': 146}
{'name': 'Aarav', 'marks': 140}


In [20]:
# 5. Update Many  and display the modified count
result = db.students.update_many({},{"$inc": {"marks": 5}})
print(f"Modified {result.modified_count} documents")

Modified 9 documents


In [24]:
# 6. Array Update
db.students.update_many({"dept":"CSE"},{"$addToSet": {"skills": "git"}})

UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [29]:
# 7. Upsert Operation
db.students.update_one(
    {"name": "Jane Smith"},
    {
        "$set": {
            "dept": "CSE",
            "marks": 85,
            "skills": ["C++", "JavaScript"],
            "city": "Bangalore",
            "fees": {
                "total": 60000,
                "paid": 60000
            }
        }
    },
    upsert=True
)

UpdateResult({'n': 1, 'upserted': ObjectId('6a87484513d922c262eb89ff'), 'nModified': 0, 'ok': 1.0, 'updatedExisting': False}, acknowledged=True)

In [ ]:
# 8. Aggregation 
pipeline = [
    {
        "$group": {
            "_id": "$dept",
            "average_marks": {"$avg": "$marks"},
            "student_count": {"$sum": 1}
        }
    },
    {
        "$sort": {"average_marks": -1}
    }
]

db.students.aggregate(pipeline)

In [ ]:
# 9. Look up :- 
pipeline = [
    {
        "$lookup": {
            "from": "departments",
            "localField": "dept",
            "foreignField": "code",
            "as": "department_info"
        }
    },
    {
        "$unwind": "$department_info"
    },
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "dept": 1,
            "hod": "$department_info.hod"
        }
    }
]

db.students.aggregate(pipeline)